# The camera model

A model trained only on all-sky camera labels, never on a citizen report. It is here for
reference, to back up the claims in `CAMERA_VALIDATION.md`; the live predictions don't use it. The
cameras sit at 58-67° magnetic latitude, which (with the 7° shift in section 2) covers observers at about 51-60°.

The notebook does four things:
1. scores the deployed citizen model (Model 1) on the camera minutes (did the citizen model learn physics, or where people report from?),
2. compares Model 1 with the cameras' aurora rate, shifting Model 1's latitude south by 0-11°,
3. fits the camera model and tests it one camera at a time,
4. scores the camera model on citizen reports it never saw (do the two models agree?).

Camera data (`camera_minutes.csv.gz`): clear, dark (sun below -12°), moon-free minutes from 9 cameras with
trustworthy labels.
- 6 THEMIS cameras (athabasca, the pas, kapuskasing, gakona, chibougamau, whitehorse), 2014-2022, labels
  from Johnson et al. (2024). Gillam, Taloyoak and Pinawa are left out: their labels drift to "always aurora" after 2019.
- Tromso (2014-2025), Kiruna (2020-2025) and Skibotn (2022-2025), labels from the Tromso AI classifier
  (Nanjo et al. 2022), aurora when arc + discrete + diffuse ≥ 80%. The 2023/24 season is left out: the aurora rate drops abruptly at all three sites at once.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from interpret.glassbox import ExplainableBoostingClassifier

from aurora_gated_model import GatedAuroraModel
from aurora_transforms import sine_mlon, cosine_mlon

pd.set_option("display.width", 200)

cams = pd.read_csv("camera_minutes.csv.gz")
cams["abs_mlat"] = cams["mlat"].abs()
cams["oval_dist"] = cams["abs_mlat"] - (66.0 - 2.0 * cams["kp"])
cams["cloud_cover"] = 0.0

order = cams.groupby("camera")["abs_mlat"].mean().sort_values().index
cams.groupby("camera").agg(mlat=("abs_mlat", "mean"), minutes=("aurora", "size"),
                           aurora_fraction=("aurora", "mean")).loc[order].round(3)

,mlat,minutes,aurora_fraction
camera,,,
chibougamau,58.433,23954,0.037
kapuskasing,58.572,95531,0.044
athabasca,61.187,122262,0.087
the pas,62.406,96407,0.132
gakona,62.930,71604,0.194
whitehorse,63.070,52677,0.407
kiruna,65.034,84322,0.702
skibotn,66.640,23830,0.841
tromso,66.768,169594,0.554


## 1. Model 1 on the camera minutes

Nobody chose these minutes, so this is the test the citizen reports can't provide. The score is the AUC within each
camera: pick one aurora minute and one quiet minute at random, and it's the chance that Model 1 gives the aurora
minute the higher number (0.5 = coin flip, 1 = perfect). A camera doesn't move, so location and
population are fixed and the ranking can only come from space weather. Cloud cover is set to 0 (the minutes are clear).

In [2]:
bundle = joblib.load("aurora_ebm_model_final_deployment.joblib")
model1, featureslow = bundle["pipeline"], bundle["featureslow"]

cams["model1"] = model1.predict_proba(cams[featureslow])[:, 1]
auc = cams.groupby("camera").apply(lambda g: roc_auc_score(g["aurora"], g["model1"]), include_groups=False)
auc = auc.loc[order].round(3)
print(auc.to_string())
print(f"\nrange {auc.min():.2f}-{auc.max():.2f}, median {auc.median():.2f}  (Model 1 on citizen reports: ~0.83)")

camera
chibougamau    0.703
kapuskasing    0.728
athabasca      0.786
the pas        0.869
gakona         0.820
whitehorse     0.704
kiruna         0.793
skibotn        0.853
tromso         0.791

range 0.70-0.87, median 0.79  (Model 1 on citizen reports: ~0.83)


## 2. The horizon shift

A camera counts aurora overhead. A person also sees it low on the northern horizon. If that matters, Model 1 at
latitude L should look like a camera at L + Δ. For each camera minute we evaluate Model 1 at the camera's latitude minus
Δ (everything else unchanged) and compare its average with the camera's aurora rate, in Kp bins.

In [3]:
kp_bins, kp_labels = [-0.1, 1.5, 3, 4.5, 9.1], ["Kp0-1", "Kp2-3", "Kp4", "Kp5+"]
deltas = [0, 3, 5, 7, 9, 11]

sample = pd.concat([g.sample(min(len(g), 40000), random_state=0) for _, g in cams.groupby("camera")])
sample["kp_bin"] = pd.cut(sample["kp"], kp_bins, labels=kp_labels)

rows = []
for d in deltas:
    x = sample.copy()
    x["mlat"] = np.sign(x["mlat"]) * (x["abs_mlat"] - d)
    x["oval_dist"] = x["mlat"].abs() - (66.0 - 2.0 * x["kp"])
    x["p"] = model1.predict_proba(x[featureslow])[:, 1]
    g = x.groupby(["camera", "kp_bin"], observed=True).agg(model1=("p", "mean"), camera_rate=("aurora", "mean"))
    g["delta"] = d
    rows.append(g.reset_index())
shift = pd.concat(rows)
shift["error"] = shift["model1"] - shift["camera_rate"]

shift.groupby("delta")["error"].agg(mean_abs_error=lambda e: e.abs().mean(), mean_bias="mean").round(3)

,mean_abs_error,mean_bias
delta,,
0,0.271,0.189
3,0.238,0.161
5,0.146,0.086
7,0.124,0.024
9,0.149,-0.086
11,0.220,-0.146


The match is best around 7°.

The same thing at Kp 4, Model 1 with no shift and with 7°, next to the camera:

In [4]:
kp4 = shift[(shift["kp_bin"] == "Kp4") & shift["delta"].isin([0, 7])]
kp4 = kp4.pivot(index="camera", columns="delta", values="model1").join(
    kp4[kp4["delta"] == 0].set_index("camera")["camera_rate"])
kp4.columns = ["model1 at camera", "model1 7° south", "camera"]
kp4.loc[order].round(2)

,model1 at camera,model1 7° south,camera
camera,,,
chibougamau,0.79,0.26,0.07
kapuskasing,0.80,0.26,0.10
athabasca,0.85,0.59,0.35
the pas,0.87,0.66,0.54
gakona,0.79,0.65,0.69
whitehorse,0.78,0.69,0.77
kiruna,0.76,0.76,0.95
skibotn,0.86,0.89,0.98
tromso,0.84,0.87,0.92


Shifting the observer about 7° poleward brings Model 1, trained on citizen reports, much closer to what the cameras record. It doesn't make them agree everywhere: 7° south of the southern cameras Model 1 is still too high, and for the Scandinavian ones too low, and the cameras differ among themselves. That's why no single correction works for all of them.

Does the best shift depend on Kp? Same comparison in 1° steps, best shift for each Kp bin and the cost of using 7° instead:

In [5]:
rows = []
for d in range(13):
    x = sample.copy()
    x["mlat"] = np.sign(x["mlat"]) * (x["abs_mlat"] - d)
    x["oval_dist"] = x["mlat"].abs() - (66.0 - 2.0 * x["kp"])
    x["p"] = model1.predict_proba(x[featureslow])[:, 1]
    g = x.groupby(["camera", "kp_bin"], observed=True).agg(model1=("p", "mean"), camera_rate=("aurora", "mean"))
    g["abs_error"] = (g["model1"] - g["camera_rate"]).abs()
    rows.append(g.groupby("kp_bin", observed=True)["abs_error"].mean().rename(d))
by_kp = pd.concat(rows, axis=1)

pd.DataFrame({"best shift": by_kp.idxmin(axis=1), "gap at best": by_kp.min(axis=1).round(3),
              "gap at 7°": by_kp[7].round(3)})

,best shift,gap at best,gap at 7°
kp_bin,,,
Kp0-1,9,0.091,0.109
Kp2-3,6,0.155,0.162
Kp4,8,0.128,0.128
Kp5+,7,0.097,0.097


The best shift moves between 6° and 9° with no trend in Kp, and using 7° everywhere widens the gap by at most 0.02. A single 7° is a reasonable choice.

## 3. The camera model

Inputs: |mlat|, Kp, Bz, solar wind speed, Ap and oval distance. No longitude, since with 9 cameras it would just learn
which camera it is. Each camera gets the same total weight, so Tromso's 170k minutes don't swamp Chibougamau's 24k.

Two versions: a logistic regression (the one quoted in `CAMERA_VALIDATION.md`) and a gradient-boosted tree, which can bend with
latitude and so, unlike the logistic regression, can respond to the 7° shift in section 4.

In [6]:
feat = ["abs_mlat", "kp", "bz_gsm", "solar_wind_speed", "Ap", "oval_dist"]
cams["w"] = 1.0 / cams.groupby("camera")["aurora"].transform("size")
cams["w"] *= len(cams) / cams["camera"].nunique()


def new_model(kind):
    if kind == "lr":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    return HistGradientBoostingClassifier(max_leaf_nodes=15, min_samples_leaf=50, l2_regularization=1.0,
                                          random_state=0)


def fit(kind, df):
    m = new_model(kind)
    if kind == "lr":
        return m.fit(df[feat], df["aurora"], logisticregression__sample_weight=df["w"])
    return m.fit(df[feat], df["aurora"], sample_weight=df["w"])

Leave one camera out: train on eight cameras, score the ninth.

In [7]:
rows = []
for c in order:
    train, test = cams[cams["camera"] != c], cams[cams["camera"] == c]
    r = {"camera": c}
    for kind in ["lr", "hgb"]:
        r[kind] = roc_auc_score(test["aurora"], fit(kind, train).predict_proba(test[feat])[:, 1])
    rows.append(r)
loco = pd.DataFrame(rows).set_index("camera").round(3)
print(loco.to_string())
print("\nmean:", loco.mean().round(3).to_dict())

                lr    hgb
camera                   
chibougamau  0.665  0.708
kapuskasing  0.730  0.708
athabasca    0.779  0.787
the pas      0.885  0.892
gakona       0.859  0.864
whitehorse   0.734  0.738
kiruna       0.820  0.823
skibotn      0.881  0.885
tromso       0.844  0.835

mean: {'lr': 0.8, 'hgb': 0.804}


The full models, trained on all nine cameras. The logistic regression's weights, back in the original units:

In [8]:
camera_lr = fit("lr", cams)
camera_hgb = fit("hgb", cams)

scaler, lr = camera_lr.named_steps["standardscaler"], camera_lr.named_steps["logisticregression"]
weights = pd.Series(lr.coef_[0] / scaler.scale_, index=feat)
intercept = lr.intercept_[0] - (lr.coef_[0] * scaler.mean_ / scaler.scale_).sum()
print(f"intercept {intercept:.2f}")
print(weights.round(4).to_string())
print(f"\nper degree of |mlat| (incl. oval_dist): {weights['abs_mlat'] + weights['oval_dist']:+.3f}")
print(f"per Kp (incl. oval_dist): {weights['kp'] + 2 * weights['oval_dist']:+.3f}")

intercept -25.51
abs_mlat            0.3723
kp                  0.3575
bz_gsm             -0.1097
solar_wind_speed   -0.0003
Ap                  0.0207
oval_dist           0.2548

per degree of |mlat| (incl. oval_dist): +0.627
per Kp (incl. oval_dist): +0.867


## 4. The camera model on citizen reports

Citizen data is cleaned and labelled exactly as in `aurora_final_deployment.ipynb`. The test set is every report after
2024-07-01. The deployed Model 1 has seen those reports, so for a fair comparison Model 1 is refit, with the same recipe,
on reports up to 2024-07-01 only. The camera models have never seen any report. Northern hemisphere only, since all
nine cameras are in the north.

For the camera models, Δ = 7 means the query uses the citizen's |mlat| + 7°: what a camera 7° further north would see.

In [9]:
data = pd.read_csv("aurora_dataset_clean_timestart.csv")

day_of_year = pd.to_datetime(data["time_start"]).dt.dayofyear
declination = np.radians(23.45 * np.sin(np.radians(360 / 365 * (day_of_year - 81))))
lat_rad = np.radians(data["latitude"])
noon_elev = np.degrees(np.arcsin(np.sin(lat_rad) * np.sin(declination) + np.cos(lat_rad) * np.cos(declination)))
polar_winter = noon_elev <= 6
dark_enough = (data["sun_elevation_localtime"] <= 6) | (data["sun_elevation_utc"] <= 6) | (data["kp"] >= 7)

cit = data.dropna(subset=["mlat", "mlon", "kp", "bz_gsm", "solar_wind_speed"]).copy()
cit["see_aurora"] = (cit["see_aurora"] == True) & (polar_winter | dark_enough)
cit = cit[(cit["solar_wind_speed"] < 9000) & (cit["bz_gsm"].abs() < 900) & (cit["sigma_bz_gsm"] < 900)
          & (cit["kp"] < 90) & (cit["ap"] < 900) & (cit["Ap"] < 900)].copy()
cit["oval_dist"] = cit["mlat"].abs() - (66.0 - 2.0 * cit["kp"])
cit["mlat_band"] = pd.cut(cit["mlat"].abs(), bins=[0, 48, 63, 90], labels=["00-48", "48-63", "63-90"])
cit["utc_dt"] = pd.to_datetime(cit["utc_dt"], format="ISO8601", utc=True)

train = cit[cit["utc_dt"] <= "2024-07-01"].copy()
test = cit[(cit["utc_dt"] > "2024-07-01") & (cit["mlat"] > 0)].copy()
print(f"train {len(train):,} reports, test {len(test):,} (northern)")

train 13,989 reports, test 7,845 (northern)


Model 1 refit on the training years (same settings as `aurora_final_deployment.ipynb`).

In [10]:
def per_band_weights(y, bands):
    y, bands = np.array(y), np.array(bands)
    w = np.ones(len(y))
    for b in np.unique(bands[~pd.isnull(bands)]):
        m = bands == b
        n_neg, n_pos, n = (y[m] == 0).sum(), (y[m] == 1).sum(), m.sum()
        if n_neg:
            w[m & (y == 0)] = n / (2 * n_neg)
        if n_pos:
            w[m & (y == 1)] = n / (2 * n_pos)
    return w


names = ["sin_mlon", "cos_mlon"] + [f for f in featureslow if f != "mlon"]
pipe = Pipeline([
    ("prepro", ColumnTransformer([("sine_mlon", FunctionTransformer(sine_mlon), ["mlon"]),
                                  ("cosine_mlon", FunctionTransformer(cosine_mlon), ["mlon"])],
                                 remainder="passthrough")),
    ("EBM", ExplainableBoostingClassifier(feature_names=names, interactions=[(3, 4), (3, 5), (4, 5)],
                                          max_bins=128, max_rounds=1000, min_samples_leaf=5, random_state=42)),
])
y = train["see_aurora"].astype(int)
w = per_band_weights(y, train["mlat_band"])
pipe.fit(train[featureslow], y, EBM__sample_weight=w)

lr_oval = LogisticRegression().fit(train[["oval_dist"]].assign(abs_mlat=train["mlat"].abs()), y, sample_weight=w)
pos = (train["sun_elevation_utc"] >= 0).values
lr_sun_pos = LogisticRegression().fit(train.loc[pos, ["sun_elevation_utc"]], y[pos], sample_weight=w[pos])
model1_train = GatedAuroraModel(pipe, lr_oval, featureslow, lr_features=["oval_dist", "abs_mlat"],
                                gate_center=-17.5, gate_scale=4.0, lr_sun_neg=None, lr_sun_pos=lr_sun_pos,
                                sun_gate_neg_center=None, sun_gate_neg_scale=None,
                                sun_gate_pos_center=6.0, sun_gate_pos_scale=10.0)

In [11]:
def camera_query(model, df, delta):
    q = df.copy()
    q["abs_mlat"] = df["mlat"].abs() + delta
    q["oval_dist"] = q["abs_mlat"] - (66.0 - 2.0 * q["kp"])
    return model.predict_proba(q[feat])[:, 1]


scores = {"Model 1 (refit on train years)": model1_train.predict_proba(test[featureslow])[:, 1]}
for name, m in [("camera lr", camera_lr), ("camera hgb", camera_hgb)]:
    for d in [0, 7]:
        scores[f"{name}, delta {d}"] = camera_query(m, test, d)

rows = []
for name, p in scores.items():
    t = test.assign(p=p)
    r = {"model": name}
    for band, g in t.groupby("mlat_band", observed=True):
        r[band] = roc_auc_score(g["see_aurora"], g["p"])
    r["all"] = roc_auc_score(t["see_aurora"], t["p"])
    rows.append(r)
pd.DataFrame(rows).set_index("model").round(3)

,00-48,48-63,63-90,all
model,,,,
Model 1 (refit on train years),0.833,0.831,0.757,0.829
"camera lr, delta 0",0.797,0.813,0.816,0.803
"camera lr, delta 7",0.797,0.813,0.816,0.803
"camera hgb, delta 0",0.667,0.755,0.738,0.740
"camera hgb, delta 7",0.748,0.772,0.702,0.767


For the logistic regression, adding 7° moves every prediction up together, so the ranking (and the AUC) can't change.
The tree can bend with latitude, and for it the shift is what lifts the score.

The exception is 63-90°, where the shift makes the tree worse. Those citizens plus 7° land at 70-97°, beyond the furthest camera (Tromso, 66.8°). Trees are bad at extrapolating: past the edge of the training data every latitude looks the same, so latitude stops helping the ranking. The logistic regression suffers less: it keeps the same slope in latitude beyond the cameras (smooth, though untested there), which is why it is the final camera model.